In [45]:
import pandas as pd

# Load the datasets
try:
    gateway_df = pd.read_csv('/content/gateway.csv')
    ledger_df = pd.read_csv('/content/ledger.csv')
    print("Successfully loaded gateway.csv and ledger.csv")
except FileNotFoundError as e:
    print(f"Error loading file: {e}. Please ensure the files are in the /content/ directory.")
    gateway_df = pd.DataFrame() # Create empty dataframe to prevent further errors
    ledger_df = pd.DataFrame() # Create empty dataframe to prevent further errors


Successfully loaded gateway.csv and ledger.csv


In [47]:
# Checking for Duplicates

if not gateway_df.empty:
    duplicate_gateway_transactions = gateway_df[gateway_df.duplicated(subset=['transaction_id'], keep=False)]
    if not duplicate_gateway_transactions.empty:
        print("Duplicate transactions found in Gateway data:")
        display(duplicate_gateway_transactions.sort_values(by='transaction_id'))
    else:
        print("No duplicate transactions found in Gateway data.")

No duplicate transactions found in Gateway data.


In [49]:
if not ledger_df.empty:
    duplicate_LEDGER_transactions = ledger_df[ledger_df.duplicated(subset=['transaction_id'], keep=False)]
    if not duplicate_LEDGER_transactions.empty:
        print("Duplicate transactions found in LEDGER data:")
        display(duplicate_LEDGER_transactions.sort_values(by='transaction_id'))
    else:
        print("No duplicate transactions found in LEDGER data.")

No duplicate transactions found in LEDGER data.


In [50]:
# Checking for NULLS

print("\nNull values in Gateway DataFrame:")
display(gateway_df.isnull().sum())




Null values in Gateway DataFrame:


,0
transaction_id,0
transaction_date,0
merchant_id,0
amount_usd,0
status,0
payment_method,0


In [52]:
print("\nNull values in LEDGER DataFrame:")
display(ledger_df.isnull().sum())


Null values in LEDGER DataFrame:


,0
transaction_id,0
transaction_date,0
merchant_id,0
amount_usd,0
status,0
payment_method,0


In [61]:
# Identifying Missing Records

if not gateway_df.empty and not ledger_df.empty:
    missing_in_ledger = gateway_df[~gateway_df['transaction_id'].isin(ledger_df['transaction_id'])]

In [58]:
if not ledger_df.empty and not gateway_df.empty:
    missing_in_GATEWAY = ledger_df[~ledger_df['transaction_id'].isin(gateway_df['transaction_id'])]

### Identify Amount Mismatches

In [76]:
if not gateway_df.empty and not ledger_df.empty:
    # Merge dataframes on transaction_id for comparison
    merged_df = pd.merge(gateway_df, ledger_df, on='transaction_id', suffixes=('_gateway', '_ledger'), how='inner')

    # Identify amount mismatches
    amount_mismatches = merged_df[merged_df['amount_usd_gateway'] != merged_df['amount_usd_ledger']]

    if not amount_mismatches.empty:
        print("Transactions with amount mismatches:")
        display(amount_mismatches[['transaction_id', 'amount_usd_gateway', 'amount_usd_ledger']])
    else:
        print("No amount mismatches found for common transactions.")
else:
    print("Cannot identify amount mismatches as one or both DataFrames are empty.")

Transactions with amount mismatches:


,transaction_id,amount_usd_gateway,amount_usd_ledger
1,R002,900.0,850.0
6,R008,600.0,640.0


### Identify Status Mismatches

In [64]:
if not gateway_df.empty and not ledger_df.empty:
    # Merge dataframes on transaction_id for comparison
    # merged_df is already defined from the previous cell, so we don't redefine it.
    # merged_df = pd.merge(gateway_df, ledger_df, on='transaction_id', suffixes=('_gateway', '_ledger'), how='inner')

    # Identify status mismatches
    status_mismatches = merged_df[merged_df['status_gateway'] != merged_df['status_ledger']]

    if not status_mismatches.empty:
        print("Transactions with status mismatches:")
        display(status_mismatches[['transaction_id', 'status_gateway', 'status_ledger']])
    else:
        print("No status mismatches found for common transactions.")
else:
    print("Cannot identify status mismatches as one or both DataFrames are empty.")

Transactions with status mismatches:


,transaction_id,status_gateway,status_ledger
3,R005,failed,success


### Create Output Directories

In [66]:
import os

output_dirs = ['01_data/processed', '04_python']
for d in output_dirs:
    os.makedirs(d, exist_ok=True)
print(f"Created directories: {', '.join(output_dirs)}")

Created directories: 01_data/processed, 04_python


### Export Discrepancy Files to CSV

In [81]:
if not missing_in_GATEWAY.empty:
    missing_in_GATEWAY.to_csv('01_data/processed/missing_in_gateway.csv', index=False)
    print("Exported missing_in_GATEWAY.csv")
else:
    print("No records missing in Gateway to export.")

if not missing_in_ledger.empty:
    missing_in_ledger.to_csv('01_data/processed/missing_in_ledger.csv', index=False)
    print("Exported missing_in_ledger.csv")
else:
    print("No records missing in Ledger to export.")

if not amount_mismatches.empty:
    amount_mismatches.to_csv('01_data/processed/amount_mismatches.csv', index=False)
    print("Exported amount_mismatches.csv")
else:
    print("No amount mismatches to export.")

if not status_mismatches.empty:
    status_mismatches.to_csv('01_data/processed/status_mismatches.csv', index=False)
    print("Exported status_mismatches.csv")
else:
    print("No status mismatches to export.")

Exported missing_in_GATEWAY.csv
Exported missing_in_ledger.csv
Exported amount_mismatches.csv
Exported status_mismatches.csv


In [82]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

### Generate Comprehensive Reconciliation Report (CSV)

In [73]:
if not gateway_df.empty and not ledger_df.empty:
    # Perform a full outer merge to capture all transactions from both datasets
    reconciliation_report_df = pd.merge(gateway_df, ledger_df, on='transaction_id', suffixes=('_gateway', '_ledger'), how='outer')

    # Add flags for different reconciliation categories
    reconciliation_report_df['in_gateway_only'] = reconciliation_report_df['transaction_date_ledger'].isnull()
    reconciliation_report_df['in_ledger_only'] = reconciliation_report_df['transaction_date_gateway'].isnull()
    reconciliation_report_df['amount_mismatch'] = (reconciliation_report_df['amount_usd_gateway'] != reconciliation_report_df['amount_usd_ledger']) & ~reconciliation_report_df['in_gateway_only'] & ~reconciliation_report_df['in_ledger_only']
    reconciliation_report_df['status_mismatch'] = (reconciliation_report_df['status_gateway'] != reconciliation_report_df['status_ledger']) & ~reconciliation_report_df['in_gateway_only'] & ~reconciliation_report_df['in_ledger_only']

    # Export the comprehensive report
    reconciliation_report_df.to_csv('01_data/processed/reconciliation_report.csv', index=False)
    print("Exported reconciliation_report.csv")
    # Display the first few rows of the report for review
    print("Reconciliation Report Head:")
    display(reconciliation_report_df.head())
else:
    print("Cannot generate reconciliation report as one or both DataFrames are empty.")

Exported reconciliation_report.csv
Reconciliation Report Head:


,transaction_id,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger,in_gateway_only,in_ledger_only,amount_mismatch,status_mismatch
0,R001,2026-03-01,M001,1200.0,success,UPI,2026-03-01,M001,1200.0,success,UPI,False,False,False,False
1,R002,2026-03-01,M002,900.0,success,Card,2026-03-01,M002,850.0,success,Card,False,False,True,False
2,R003,2026-03-02,M001,500.0,success,Wallet,2026-03-02,M001,500.0,success,Wallet,False,False,False,False
3,R004,NaN,NaN,NaN,NaN,NaN,2026-03-02,M003,2100.0,success,Card,False,True,False,False
4,R005,2026-03-03,M004,7200.0,failed,Card,2026-03-03,M004,7200.0,success,Card,False,False,False,True


### Export Summary Metrics to JSON

In [83]:
import json

# Calculate additional metrics
total_gateway_rows = len(gateway_df) if not gateway_df.empty else 0
total_ledger_rows = len(ledger_df) if not ledger_df.empty else 0
missing_in_gateway_count = len(missing_in_GATEWAY) if 'missing_in_GATEWAY' in locals() and not missing_in_GATEWAY.empty else 0
missing_in_ledger_count = len(missing_in_ledger) if 'missing_in_ledger' in locals() and not missing_in_ledger.empty else 0
amount_mismatch_count = len(amount_mismatches) if 'amount_mismatches' in locals() and not amount_mismatches.empty else 0
status_mismatch_count = len(status_mismatches) if 'status_mismatches' in locals() and not status_mismatches.empty else 0

reconciliation_issue_count = (
    missing_in_gateway_count +
    missing_in_ledger_count +
    amount_mismatch_count +
    status_mismatch_count
)

ledger_total_amount = ledger_df['amount_usd'].sum() if not ledger_df.empty else 0
gateway_total_amount = gateway_df['amount_usd'].sum() if not gateway_df.empty else 0

# Calculate amount at risk (absolute difference for mismatched amounts)
if 'amount_mismatches' in locals() and not amount_mismatches.empty:
    amount_at_risk = (amount_mismatches['amount_usd_gateway'] - amount_mismatches['amount_usd_ledger']).abs().sum()
else:
    amount_at_risk = 0

summary_data = {
  "total_ledger_rows": total_ledger_rows,
  "total_gateway_rows": total_gateway_rows,
  "missing_in_gateway_count": missing_in_gateway_count,
  "missing_in_ledger_count": missing_in_ledger_count,
  "amount_mismatch_count": amount_mismatch_count,
  "status_mismatch_count": status_mismatch_count,
  "reconciliation_issue_count": reconciliation_issue_count,
  "ledger_total_amount": ledger_total_amount,
  "gateway_total_amount": gateway_total_amount,
  "amount_at_risk": amount_at_risk
}

with open('04_python/summary_metrics.json', 'w') as f:
    json.dump(summary_data, f, indent=4)
print("Exported summary_metrics.json")

Exported summary_metrics.json


In [85]:
import json

# Display the contents of the summary_data dictionary
print(json.dumps(summary_data, indent=4))

{
    "total_ledger_rows": 10,
    "total_gateway_rows": 9,
    "missing_in_gateway_count": 2,
    "missing_in_ledger_count": 1,
    "amount_mismatch_count": 2,
    "status_mismatch_count": 1,
    "reconciliation_issue_count": 6,
    "ledger_total_amount": 23340.0,
    "gateway_total_amount": 20550.0,
    "amount_at_risk": 90.0
}


In [86]:
import json
import pandas as pd

# Define the path to the nested JSON file
json_file_path = '/content/api_response_sample.json'

# Load the JSON data
with open(json_file_path, 'r') as f:
    nested_json_data = json.load(f)

# Flatten the nested JSON data using json_normalize
# Assuming the main data is under a key, or it's a list of records
# Let's inspect the structure if needed by printing nested_json_data.keys() or nested_json_data[0].keys()
# For now, assuming it's directly a list of records or a dictionary containing a list of records.
# Based on the typical structure of API responses, often the core data is in a key like 'data' or 'records'.
# If it's a list directly, json_normalize can take it as is.

# Heuristic: if it's a dictionary and has a single key whose value is a list, try that key.
# Otherwise, assume it's a list or directly flattens from the top level.

if isinstance(nested_json_data, dict) and len(nested_json_data) == 1:
    # Get the key of the dictionary (e.g., 'data' or 'items')
    main_key = list(nested_json_data.keys())[0]
    if isinstance(nested_json_data[main_key], list):
        df_flattened = pd.json_normalize(nested_json_data[main_key])
    else:
        df_flattened = pd.json_normalize(nested_json_data)
elif isinstance(nested_json_data, list):
    df_flattened = pd.json_normalize(nested_json_data)
else:
    df_flattened = pd.json_normalize(nested_json_data)

print("Flattened DataFrame head:")
display(df_flattened.head())

Flattened DataFrame head:


,generated_at,source,batches
0,2026-03-07T10:00:00Z,QuickPay Settlement API,"[{'batch_id': 'B001', 'merchant': {'merchant_i..."


In [87]:
# Clean column names: replace '.' with '_' and convert to lowercase
def clean_column_names(df):
    df.columns = df.columns.str.replace('.', '_', regex=False).str.lower()
    return df

df_cleaned = clean_column_names(df_flattened)

print("DataFrame with cleaned column names:")
display(df_cleaned.head())

DataFrame with cleaned column names:


,generated_at,source,batches
0,2026-03-07T10:00:00Z,QuickPay Settlement API,"[{'batch_id': 'B001', 'merchant': {'merchant_i..."


In [90]:
# Identify and convert potential date/time fields
# This step requires inspecting the column names from df_cleaned.head() output
# For demonstration, let's assume 'timestamp' or 'date' are common date/time fields.

# Check available columns to identify date-like columns
date_cols_to_convert = []
print("Columns in df_cleaned:", df_cleaned.columns.tolist())

# Ensure 'generated_at' is included if it exists, as it's a known date/time field from inspection
if 'generated_at' in df_cleaned.columns:
    date_cols_to_convert.append('generated_at')
    print("Explicitly added 'generated_at' to date_cols_to_convert.")

# Continue with general pattern matching for other potential date/time columns
for col in df_cleaned.columns:
    if col == 'generated_at': # Skip if already explicitly added
        continue
    # Common patterns for date/time columns
    if 'date' in col or 'time' in col or 'created_at' in col or 'updated_at' in col:
        date_cols_to_convert.append(col)
        print(f"Added '{col}' to date_cols_to_convert based on pattern match.")

print(f"Final list of columns to convert: {date_cols_to_convert}")

# Attempt to convert identified columns to datetime
for col in date_cols_to_convert:
    if col in df_cleaned.columns:
        try:
            df_cleaned[col] = pd.to_datetime(df_cleaned[col], errors='coerce')
            print(f"Converted column '{col}' to datetime.")
        except Exception as e:
            print(f"Could not convert column '{col}' to datetime: {e}")

print("DataFrame info after date/time conversion:")
df_cleaned.info()

Columns in df_cleaned: ['generated_at', 'source', 'batches']
Explicitly added 'generated_at' to date_cols_to_convert.
Final list of columns to convert: ['generated_at']
Converted column 'generated_at' to datetime.
DataFrame info after date/time conversion:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   generated_at  1 non-null      datetime64[ns, UTC]
 1   source        1 non-null      object             
 2   batches       1 non-null      object             
dtypes: datetime64[ns, UTC](1), object(2)
memory usage: 156.0+ bytes


In [89]:
# Save the normalized output to a CSV file
output_csv_path = '01_data/processed/normalized_api_response.csv'
df_cleaned.to_csv(output_csv_path, index=False)

print(f"Normalized data saved to: {output_csv_path}")

Normalized data saved to: 01_data/processed/normalized_api_response.csv
